In [1]:
# 캐시 지우기
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel,TextStreamer, pipeline 
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA, LLMChain
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
import torch
import pandas as pd

In [3]:
# 검색 관련 함수
def extract_company_info(company_name):
    # 크롬 옵션 설정
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    # WebDriver 설정
    driver = webdriver.Chrome(options=chrome_options)

    try:
        # 변수 설정
        base_url = "https://www.jobkorea.co.kr/starter/companyreport"

        # URL 구성 및 접속
        url = f"{base_url}?schTxt={company_name}"
        driver.get(url)

        # 원하는 요소가 로드될 때까지 기다리고 요소 찾기
        element = WebDriverWait(driver, 3).until(
            EC.presence_of_element_located((By.XPATH, f'//dt/a/strong[contains(text(), "기업심층분석 1. {company_name}, 채용분석 및 기업정보")]'))
        )
    
        href = element.find_element(By.XPATH, '..').get_attribute('href')
        print(f"Found detailed analysis link: {href}")

        # 링크로 이동
        driver.get(href)

        # 페이지 소스 가져오기
        page_source = driver.page_source

        # BeautifulSoup으로 페이지 소스 파싱
        soup = BeautifulSoup(page_source, 'html.parser')

        # 'viewWrap' 클래스를 가진 모든 요소 찾기
        view_wrap_elements = soup.find_all(class_="viewWrap")

        # 각 요소에서 텍스트 추출
        extracted_text = ""
        for element in view_wrap_elements:
            text = element.get_text(strip=True)
            extracted_text += text + "\n"
            
        print(extracted_text)
        return extracted_text
        

    except Exception as e:
        print(f"An error occurred: {e}")
        return None

    finally:
        driver.quit()

In [4]:
# Orion-14B 모델 및 토크나이저 로드
model_orion = AutoModelForCausalLM.from_pretrained("cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0", torch_dtype=torch.float16, trust_remote_code=True, device_map="auto")
tokenizer_orion = AutoTokenizer.from_pretrained("cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0", use_fast=False, trust_remote_code=True)
streamer = TextStreamer(tokenizer_orion)
# BGE-M3 모델 및 토크나이저 로드
# model_bge = AutoModel.from_pretrained('BAAI/bge-m3',device_map="auto")
# tokenizer_bge = AutoTokenizer.from_pretrained('BAAI/bge-m3')
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": "cuda:5"})

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

TypeError: not a string

In [ ]:
def generate_talent_description(company_name, retriever,llm):
    # 질의 생성
    query = f""" 
    "회사의 모든 인재상을 입력하세요."
    "인재상 설명 이외의 다른 단어는 출력하지 마세요."
    "질문에 최선을 다해 답변하세요. "
    "검색 가능한 모든 도구를 자유롭게 사용하세요 "
    "필요한 경우에만 관련 정보 제공을 하세요"
    "답을 모르면 모른다고 말하세요. 답을 지어내려고 하지 마세요."
    "반드시 한국어로 답변하세요"
    """
    
    # 답변 생성
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )
    
    result = qa_chain({"query": query})
    talents = result["result"]
    
    # "Helpful Answer:" 이후의 텍스트만 추출
    helpful_answer_index = talents.find("Helpful Answer:")
    if helpful_answer_index != -1:
        talents = talents[helpful_answer_index + len("Helpful Answer:"):].strip()

    return talents

In [ ]:

# 변수 설정
company_name = "삼성SDS"
report = """
안녕하세요. 저는 호남대학교 컴퓨터공학과 4학년에 재학 중인 이진수입니다. 저는 어려서부터 컴퓨터와 프로그래밍에 관심이 많았고, 대학에 입학한 후에는 소프트웨어 개발에 필요한 다양한 기술과 지식을 배우며 역량을 키워 왔습니다.
저는 학부 과정에서 프로그래밍 언어, 자료구조, 알고리즘, 데이터베이스, 웹 개발 등 컴퓨터공학의 기본 지식을 충실히 학습했습니다. 특히 Java와 Python을 활용한 프로젝트를 통해 실무에 적용 가능한 개발 기술을 연마했습니다. 3학년 때는 '학생 관리 시스템' 프로젝트에 참여하여 웹 애플리케이션 개발 전 과정을 경험하였고, 백엔드 개발을 담당하며 REST API 설계와 데이터베이스 관리 능력을 키웠습니다.
또한, 저는 꾸준한 자기개발을 통해 최신 기술 동향을 파악하고 역량을 강화하고 있습니다. 최근에는 클라우드 컴퓨팅과 머신러닝에 관심을 갖고 AWS와 TensorFlow를 학습하고 있습니다. 이러한 새로운 기술을 프로젝트에 적극적으로 활용하여 혁신적인 솔루션을 개발하는 것이 저의 목표입니다.
"""
# CSV 파일 경로
csv_file_path = "./company_talents.csv"

# CSV 파일 읽기
df = pd.read_csv(csv_file_path)

# 파이프라인 객체 생성
hf_pipeline = pipeline("text-generation", model=model_orion, tokenizer=tokenizer_orion, streamer=streamer)
    
# HuggingFacePipeline으로 모델 래핑
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# company_name 열에 company_name이 있는지 확인
if company_name in df["company_name"].values:
    # company_name이 존재하는 경우, talents 열의 값을 가져옴
    talents = df[df["company_name"] == company_name]["talents"].tolist()
    # 결과 출력
    print("Talents:", talents)
else:
    # company_name이 존재하지 않는 경우, extract_company_info 함수 실행
    company_info = extract_company_info(company_name)
    if company_info:
        # 문서 임베딩
        vector_store = FAISS.from_texts(texts=[company_info], embedding=embeddings)

        # Retriever 생성
        retriever = vector_store.as_retriever(search_kwargs={"k": 1})

        # 인재상 생성
        talents = generate_talent_description(company_name, retriever, llm)
        print(talents)
        print(type(talents))
        str(talents)

        # 결과를 데이터프레임에 추가
        new_data = {"company_name": company_name, "talents": talents}
        df = pd.concat([df, pd.DataFrame([new_data])], ignore_index=True)
        
        # 결과를 CSV 파일에 저장
        df.to_csv(csv_file_path, index=False)
        print("인재상이 생성되어 CSV 파일에 추가되었습니다.")
    else:
        print("기업 정보 추출에 실패했습니다. 직접 인재상을 작성해주세요")
        # 인재상 입력
        talents = input("인재상 : ")
        
        # 결과를 데이터프레임에 추가
        new_data = {"company_name": company_name, "talents": talents}
        df = pd.concat([df, pd.DataFrame([new_data])], ignore_index=True)

        # 결과를 CSV 파일에 저장
        df.to_csv(csv_file_path, index=False)
        print("인재상이 생성되어 CSV 파일에 추가되었습니다.")

Talents: ['"삼성SDS의 인재상은 다음과 같습니다:\n    - 열정(Passion): 끊임없는 열정으로 미래에 도전하는 인재\n    - 창의혁신(Creativity): 창의와 혁신으로 세상을 변화시키는 인재\n    - 인간미·도덕성(Integrity): 정직과 바른 행동으로 역할과 책임을 다하는 인재\n    \n    삼성SDS는 이러한 인재상을 바탕으로 다양한 산업부문에서 디지털 혁신을 선도하고 있습니다."']


In [ ]:
# 데이터 삭제 요청
company_name = "BCD"

if company_name in df["company_name"].values:
    # company_name이 존재하는 경우, talents 열의 값을 가져옴
    talents = df[df["company_name"] == company_name]["talents"].tolist()
    # 결과 출력
    print("Talents:", talents)
    
    # company_name 행 제거
    df = df[df["company_name"] != company_name]
    
    # 수정된 데이터프레임을 CSV 파일에 저장
    df.to_csv(csv_file_path, index=False)
    print(f"{company_name} 행이 CSV 파일에서 제거되었습니다.")
else:
    print("CSV에 없는 회사명입니다. 다시한번 확인해주세요")

CSV에 없는 회사명입니다. 다시한번 확인해주세요


In [ ]:
# # 프롬프트 정의
# prompt_template = """
# 당신은 인재상을 기준으로 사용자의 "자기소개서"를 수정하는 역할을 맡고 있습니다. 아래의 첨삭과정에에 따라 자기소개서 생성을 수행하세요.
# 생성은 아래의 순서대로 진행해주세요.

# [첨삭 과정]
# 1. [기업명]과 [인재상]을 확인하세요.
# 2. 자기소개서의 전체 내용을 파악한 후, 나의 인성 및 역량에서 해당되는 부분을 매칭하고 인재상을 자연스럽게 녹여내세요. 
# 3. 인재상을 한 부분에 몰아서 넣지 않도록 하세요. 마지막에 인재상을 몰아서 작성하지 마세요.
# 4. 기업의 이름과 관련된 고유명사 모두 'OOO'로 대체하세요.
# 5. 답변은 한국어로 작성하세요.
# 6. 출력 형식은 반드시 아래와 같이 유지하세요.

# [출력형식]
# 수정된 자기소개서: (수정된 자기소개서 내용)
# """

# # 입력값 정의
# input_template = """
# [기업명]
# {company_name}

# [인재상]
# {talents}

# [자기소개서]
# {report}

# 위 [인재상]을 참고하여 [자기소개서]를 [첨삭과정]에 따라 수정해주세요. 출력 형식을 반드시 지켜주세요.
# """

# # 프롬프트와 입력값 결합
# prompt = prompt_template + input_template.format(company_name=company_name, talents=talents, report=report)

# # 언어 모델 실행
# result = llm(prompt,templature=1.7)
# print(result)

In [ ]:
prompt_template = f"""
당신은 인재상을 기준으로 사용자의 자기소개서를 수정하는 역할을 맡고 있습니다. 아래의 첨삭과정을 지키면서 작업을 수행하세요.

[입력 정보]
기업명: {company_name}
인재상: {talents}
자기소개서: {report}

[첨삭 과정]
1. {company_name}의 인재상을 확인하세요.
2. 제공된 자기소개서의 내용과 흐름을 분석하세요.
3. 자기소개서에서 지원자의 인성 및 역량 중 인재상과 부합하는 부분을 식별하세요.
4. 식별된 부분을 강조하고, 해당 문단에서 인재상과 관련된 경험이나 역량을 구체적으로 설명하세요.
5. 모든 문단에 걸쳐 인재상이 고르게 반영되도록 하세요. 
6. 한 문단에 인재상 관련 내용이 집중되지 않도록 주의하세요.
7. 기업명 '{company_name}'을 포함한 모든 고유명사를 'OOO'로 대체하세요.
8. 수정된 문단들을 하나의 자기소개서로 통합하세요.

[출력 형식]
수정된 자기소개서: (수정된 자기소개서 내용)
"""

# 언어 모델 실행
result = llm(prompt=prompt_template,templature=1.2)
print(result)


당신은 인재상을 기준으로 사용자의 자기소개서를 수정하는 역할을 맡고 있습니다. 아래의 첨삭과정을 지키면서 작업을 수행하세요.

[입력 정보]
기업명: 삼성SDS
인재상: ['"삼성SDS의 인재상은 다음과 같습니다:\n    - 열정(Passion): 끊임없는 열정으로 미래에 도전하는 인재\n    - 창의혁신(Creativity): 창의와 혁신으로 세상을 변화시키는 인재\n    - 인간미·도덕성(Integrity): 정직과 바른 행동으로 역할과 책임을 다하는 인재\n    \n    삼성SDS는 이러한 인재상을 바탕으로 다양한 산업부문에서 디지털 혁신을 선도하고 있습니다."']
자기소개서: 
안녕하세요. 저는 호남대학교 컴퓨터공학과 4학년에 재학 중인 이진수입니다. 저는 어려서부터 컴퓨터와 프로그래밍에 관심이 많았고, 대학에 입학한 후에는 소프트웨어 개발에 필요한 다양한 기술과 지식을 배우며 역량을 키워 왔습니다.
저는 학부 과정에서 프로그래밍 언어, 자료구조, 알고리즘, 데이터베이스, 웹 개발 등 컴퓨터공학의 기본 지식을 충실히 학습했습니다. 특히 Java와 Python을 활용한 프로젝트를 통해 실무에 적용 가능한 개발 기술을 연마했습니다. 3학년 때는 '학생 관리 시스템' 프로젝트에 참여하여 웹 애플리케이션 개발 전 과정을 경험하였고, 백엔드 개발을 담당하며 REST API 설계와 데이터베이스 관리 능력을 키웠습니다.
또한, 저는 꾸준한 자기개발을 통해 최신 기술 동향을 파악하고 역량을 강화하고 있습니다. 최근에는 클라우드 컴퓨팅과 머신러닝에 관심을 갖고 AWS와 TensorFlow를 학습하고 있습니다. 이러한 새로운 기술을 프로젝트에 적극적으로 활용하여 혁신적인 솔루션을 개발하는 것이 저의 목표입니다.


[첨삭 과정]
1. 삼성SDS의 인재상을 확인하세요.
2. 제공된 자기소개서의 내용과 흐름을 분석하세요.
3. 자기소개서에서 지원자의 인성 및 역량 중 인재상과 부합하는 부분을 식별하세요.
4. 식별된 부분을 강조하고, 해당 문단에서 인재상과 관련된

/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/transformers/pipelines/base.py:1157: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(


수정된 자기소개서: 
안녕하세요. OOO에 지원한 OOO입니다. 저는 어려서부터 컴퓨터와 프로그래밍에 관심이 많았고, 대학에 입학한 후에는 소프트웨어 개발에 필요한 다양한 기술과 지식을 배우며 역량을 키워 왔습니다.

저는 학부 과정에서 프로그래밍 언어, 자료구조, 알고리즘, 데이터베이스, 웹 개발 등 컴퓨터공학의 기본 지식을 충실히 학습했습니다. 특히 Java와 Python을 활용한 프로젝트를 통해 실무에 적용 가능한 개발 기술을 연마했습니다. 3학년 때는 '학생 관리 시스템' 프로젝트에 참여하여 웹 애플리케이션 개발 전 과정을 경험하였고, 백엔드 개발을 담당하며 REST API 설계와 데이터베이스 관리 능력을 키웠습니다.

또한, 저는 꾸준한 자기개발을 통해 최신 기술 동향을 파악하고 역량을 강화하고 있습니다. 최근에는 클라우드 컴퓨팅과 머신러닝에 관심을 갖고 AWS와 TensorFlow를 학습하고 있습니다. 이러한 새로운 기술을 프로젝트에 적극적으로 활용하여 혁신적인 솔루션을 개발하는 것이 저의 목표입니다.

저는 OOO의 인재상 중 '창의혁신(Creativity)'과 '인간미·도덕성(Integrity)'에 부합하는 역량과 경험을 보유하고 있다고 생각합니다. 학부 시절, 저는 창의적인 아이디어로 '학생 관리 시스템' 프로젝트를 성공적으로 이끌었으며, 정직과 바른 행동을 중요시하며 팀원들과 원활한 협업을 이끌어냈습니다.

또한, 저는 OOO의 열정(Passion)에 공감하며, 디지털 혁신을 선도하는 OOO에서 제 역량을 마음껏 발휘하고  싶습니다.</s>

당신은 인재상을 기준으로 사용자의 자기소개서를 수정하는 역할을 맡고 있습니다. 아래의 첨삭과정을 지키면서 작업을 수행하세요.

[입력 정보]
기업명: 삼성SDS
인재상: ['"삼성SDS의 인재상은 다음과 같습니다:\n    - 열정(Passion): 끊임없는 열정으로 미래에 도전하는 인재\n    - 창의혁신(Creativity): 창의와 혁신으로 세상을 변화시키는 인재\n    - 인간미·도덕성(Int